# Multi-Agent Systems

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangChain roadmap.

You will learn how to build multi-agent systems using handoffs, routers, subagents, and skills patterns.

## 1. Install Dependencies

In [ ]:
!pip install -q langchain "langchain[openai]" langgraph langchain-mcp-adapters

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Create Specialist Agents

Define agents with distinct roles — a billing specialist and a tech support specialist.

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent

model = init_chat_model("gpt-4o-mini", model_provider="openai")

billing_agent = create_react_agent(
    model=model,
    tools=[],
    name="billing_agent",
    prompt="You are a billing specialist. Help users with invoices, payments, and account charges.",
)

tech_agent = create_react_agent(
    model=model,
    tools=[],
    name="tech_agent",
    prompt="You are a technical support specialist. Help users with software issues and troubleshooting.",
)

print("Specialist agents created.")

## 4. Handoffs Pattern

A triage agent routes conversations to the right specialist using handoffs.

In [ ]:
triage_agent = create_react_agent(
    model=model,
    tools=[],
    name="triage_agent",
    prompt="You are a triage agent. Route billing questions to billing_agent and technical questions to tech_agent.",
    handoffs=["billing_agent", "tech_agent"],
)

In [ ]:
from langchain_core.messages import HumanMessage

result = triage_agent.invoke({
    "messages": [HumanMessage(content="I was charged twice on my last invoice")]
})
print(result["messages"][-1].content)

In [ ]:
result = triage_agent.invoke({
    "messages": [HumanMessage(content="My app crashes when I click the submit button")]
})
print(result["messages"][-1].content)

## 5. Router Pattern

Classify input with a tool and dispatch to the correct specialist.

In [ ]:
from langchain_core.tools import tool

@tool
def classify_intent(message: str) -> str:
    """Classify the user's intent into a category."""
    message_lower = message.lower()
    if any(word in message_lower for word in ["bill", "invoice", "payment", "charge"]):
        return "billing"
    elif any(word in message_lower for word in ["bug", "error", "crash", "install"]):
        return "technical"
    return "general"

router_agent = create_react_agent(
    model=model,
    tools=[classify_intent],
    name="router",
    prompt=(
        "You are a router. Use classify_intent to determine the category, "
        "then hand off to the appropriate specialist."
    ),
    handoffs=["billing_agent", "tech_agent"],
)

result = router_agent.invoke({
    "messages": [HumanMessage(content="How do I pay my outstanding balance?")]
})
print(result["messages"][-1].content)

## 6. Subagents Pattern

A parent agent spawns child agents for independent subtasks.

In [ ]:
@tool
def research_topic(topic: str) -> str:
    """Research a topic using a dedicated subagent."""
    research_agent = create_react_agent(
        model=model,
        tools=[],
        prompt=f"Research the following topic thoroughly: {topic}",
    )
    result = research_agent.invoke({
        "messages": [{"role": "user", "content": f"Research: {topic}"}]
    })
    return result["messages"][-1].content

parent_agent = create_react_agent(
    model=model,
    tools=[research_topic],
    prompt="You are a research coordinator. Break complex questions into subtopics and research each one.",
)

result = parent_agent.invoke({
    "messages": [HumanMessage(content="What are the pros and cons of microservices?")]
})
print(result["messages"][-1].content)

## 7. Skills Pattern

Load specialized prompts on-demand based on the task.

In [ ]:
SKILLS = {
    "code_review": "You are a code reviewer. Analyze code for bugs, style issues, and improvements.",
    "sql_expert": "You are a SQL expert. Write optimized queries and explain execution plans.",
    "writer": "You are a technical writer. Produce clear, concise documentation.",
}

@tool
def activate_skill(skill_name: str, task: str) -> str:
    """Activate a specialized skill to handle a specific task."""
    skill_prompt = SKILLS.get(skill_name)
    if not skill_prompt:
        return f"Unknown skill: {skill_name}. Available: {list(SKILLS.keys())}"

    skill_agent = create_react_agent(
        model=model,
        tools=[],
        prompt=skill_prompt,
    )
    result = skill_agent.invoke({
        "messages": [{"role": "user", "content": task}]
    })
    return result["messages"][-1].content

skill_agent = create_react_agent(
    model=model,
    tools=[activate_skill],
    prompt="You have access to specialized skills: code_review, sql_expert, writer. Activate the right skill for each task.",
)

result = skill_agent.invoke({
    "messages": [HumanMessage(content="Write a SQL query to find the top 10 customers by revenue")]
})
print(result["messages"][-1].content)

## Summary

- **Handoffs** transfer control between agents using the `handoffs` parameter
- **Routers** classify input and dispatch to the right specialist
- **Subagents** spawn child agents for independent subtasks
- **Skills** inject specialized prompts on-demand
- Start with handoffs for simple routing; use subagents for parallel decomposition